In [11]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

%load_ext autoreload
%autoreload 2
from preprocess import TitanicPreprocessor

OG_TRAIN_PATH = "../data/original/train.csv"
OG_TEST_PATH = "../data/original/test.csv"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
def get_nullcnt_ratio_describe(df: pd.DataFrame) -> pd.DataFrame:
    """주어진 DataFrame 객체에서 결측치가 존재하는 컬럼의 이름과 그 개수, 비율, 그리고 정보를 담은 DataFrame을 리턴합니다."""
    # 각 컬럼별 결측치 개수 계산
    null_counts = df.isnull().sum()
    
    # 결측치가 1개 이상 존재하는 컬럼만 필터링
    null_counts = null_counts[null_counts > 0]
    
    # 결측치 비율 계산 (결측치 개수 / 전체 행 개수)
    null_ratios = null_counts / len(df)

    # 결측치 행의 정보 확인
    null_infos = df[null_counts.index].describe(include='all').transpose()
    
    # 데이터프레임으로 변환
    null_df = pd.DataFrame({
        'Null_Count': null_counts,
        'Null_Ratio': null_ratios,
    })
    null_df = pd.concat([null_df, null_infos], axis=1)
    
    # 결측치 개수를 기준으로 내림차순 정렬하여 반환
    return null_df.sort_values(by='Null_Count', ascending=False)

In [13]:
preprocessor = TitanicPreprocessor()

train_df, test_df = preprocessor.preprocess(OG_TRAIN_PATH, OG_TEST_PATH)

print("========================================================================")
print("Original train_df shape:", preprocessor.og_train_df.shape)
print("Original train_df columns:", preprocessor.og_train_df.columns)
print("------------------------------------------------------------------------")
print("Preprocessed train_df shape:", train_df.shape)
print("Preprocessed train_df columns:", train_df.columns)
print("------------------------------------------------------------------------")
print("Added columns:", set(train_df.columns) - set(preprocessor.og_train_df.columns))
print("========================================================================")
print("Original test_df shape:", preprocessor.og_test_df.shape)
print("Original test_df columns:", preprocessor.og_test_df.columns)
print("------------------------------------------------------------------------")
print("Preprocessed test_df shape:", test_df.shape)
print("Preprocessed test_df columns:", test_df.columns)
print("------------------------------------------------------------------------")
print("Added columns:", set(test_df.columns) - set(preprocessor.og_test_df.columns))
print("========================================================================")

print("Preprocessed train_df head:")
display(train_df.head(3))
print("Preprocessed test_df head:")
display(test_df.head(3))

Original train_df shape: (891, 12)
Original train_df columns: Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='str')
------------------------------------------------------------------------
Preprocessed train_df shape: (891, 18)
Preprocessed train_df columns: Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked', 'hasAge', 'hasCabin',
       'Title', 'Deck', 'Side', 'Companion'],
      dtype='str')
------------------------------------------------------------------------
Added columns: {'Deck', 'Side', 'Title', 'Companion', 'hasAge', 'hasCabin'}
Original test_df shape: (418, 11)
Original test_df columns: Index(['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch',
       'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='str')
------------------------------------------------------------------------
Preproce

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,hasAge,hasCabin,Title,Deck,Side,Companion
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,1,0,Mr,Unknown,Unknown,1
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,1,1,Mrs,C,Starboard,1
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1,0,Miss,Unknown,Unknown,0


Preprocessed test_df head:


,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,hasAge,hasCabin,Title,Deck,Side,Companion
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q,1,0,Mr,Unknown,Unknown,0
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S,1,0,Mrs,Unknown,Unknown,1
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q,1,0,Mr,Unknown,Unknown,0


In [14]:
print("train_df 결측치 정보:")
display(get_nullcnt_ratio_describe(train_df))
print("test_df 결측치 정보:")
display(get_nullcnt_ratio_describe(test_df))

train_df 결측치 정보:


,Null_Count,Null_Ratio,count,unique,top,freq
Cabin,687,0.771044,204,147,G6,4


test_df 결측치 정보:


,Null_Count,Null_Ratio,count,unique,top,freq
Cabin,327,0.782297,91,76,B57 B59 B63 B66,3


In [ ]:
print("preprocessed train_df info:")
display(train_df.info())